In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.animation as animation
from IPython.display import HTML
import heapq
from collections import defaultdict

PROJECT_DIR = '/content/drive/MyDrive/FAB_AMR_Project'
SIM_DIR     = f'{PROJECT_DIR}/04_simulator'
OUTPUT_DIR  = f'{SIM_DIR}/output'

print("✅ 준비 완료!")

Mounted at /content/drive
✅ 준비 완료!


In [2]:
grid_map = np.load(f'{OUTPUT_DIR}/grid_map.npy')

with open(f'{OUTPUT_DIR}/port_locations.json') as f:
    port_locations = json.load(f)

with open(f'{OUTPUT_DIR}/fab_layout.json') as f:
    fab_layout = json.load(f)

grid_h, grid_w = grid_map.shape
port_names = list(port_locations.keys())

print(f"격자 크기: {grid_w} x {grid_h}")
print(f"포트 수:   {len(port_names)}개")
print("✅ 로드 완료!")

격자 크기: 159 x 86
포트 수:   1116개
✅ 로드 완료!


In [3]:
def astar(grid_map, start, goal):
    grid_h, grid_w = grid_map.shape

    def heuristic(a, b):
        return abs(a[0]-b[0]) + abs(a[1]-b[1])

    def is_valid(x, y):
        return (0 <= x < grid_w and
                0 <= y < grid_h and
                grid_map[y][x] != 1)

    directions = [(0,1),(0,-1),(1,0),(-1,0),
                  (1,1),(1,-1),(-1,1),(-1,-1)]
    dir_costs  = [1,1,1,1,1.414,1.414,1.414,1.414]

    if start == goal:
        return [start]
    if not is_valid(*start) or not is_valid(*goal):
        return None

    open_set = []
    heapq.heappush(open_set, (0, 0, start))
    came_from = {}
    g_cost = {start: 0}

    while open_set:
        _, current_g, current = heapq.heappop(open_set)

        if current == goal:
            path = []
            node = goal
            while node in came_from:
                path.append(node)
                node = came_from[node]
            path.append(start)
            path.reverse()
            return path

        if current_g > g_cost.get(current, float('inf')):
            continue

        for (dx, dy), cost in zip(directions, dir_costs):
            nb = (current[0]+dx, current[1]+dy)
            if not is_valid(*nb):
                continue
            new_g = g_cost[current] + cost
            if new_g < g_cost.get(nb, float('inf')):
                g_cost[nb] = new_g
                came_from[nb] = current
                heapq.heappush(
                    open_set,
                    (new_g + heuristic(nb, goal), new_g, nb)
                )
    return None

print("✅ A* 함수 정의 완료!")

✅ A* 함수 정의 완료!


In [4]:
# Resevation Table 기반 충돌 회피
class ReservationTable:
    """
    시간-공간 예약 테이블
    어떤 AMR이 어떤 시간에 어떤 셀을 점유하는지 관리

    충돌 유형 2가지:
    1. Vertex collision: 같은 시간 같은 셀
    2. Edge collision:   서로 위치 교환 (맞바꿈)
    """

    def __init__(self):
        # {(x, y, t): amr_id}
        self.vertex_table = {}
        # {(x1,y1,x2,y2,t): amr_id}
        self.edge_table   = {}

    def is_vertex_conflict(self, x, y, t):
        """해당 셀이 시간 t에 이미 예약됐는지"""
        return (x, y, t) in self.vertex_table

    def is_edge_conflict(self, x1, y1, x2, y2, t):
        """두 AMR이 서로 위치를 맞바꾸는지"""
        return (x2, y2, x1, y1, t) in self.edge_table

    def reserve(self, path, amr_id):
        """경로 전체를 예약 테이블에 등록"""
        for t, (x, y) in enumerate(path):
            self.vertex_table[(x, y, t)] = amr_id
            if t > 0:
                px, py = path[t-1]
                self.edge_table[(px, py, x, y, t)] = amr_id

    def clear(self):
        self.vertex_table.clear()
        self.edge_table.clear()


def astar_with_reservation(grid_map, start, goal,
                             reservation, amr_id,
                             max_time=500):
    """
    Reservation Table을 고려한 A* 경로 탐색
    시간 축을 추가해서 (x, y, t) 공간에서 탐색
    """
    grid_h, grid_w = grid_map.shape

    def is_valid(x, y):
        return (0 <= x < grid_w and
                0 <= y < grid_h and
                grid_map[y][x] != 1)

    def heuristic(x, y):
        return abs(x - goal[0]) + abs(y - goal[1])

    directions = [(0,0),   # 대기 (wait)
                  (0,1),(0,-1),(1,0),(-1,0),
                  (1,1),(1,-1),(-1,1),(-1,-1)]
    dir_costs  = [0.5, 1,1,1,1, 1.414,1.414,1.414,1.414]
    # 대기 비용을 0.5로 설정 → 너무 많이 기다리지 않도록

    # 시작 상태: (x, y, t=0)
    start_state = (start[0], start[1], 0)
    open_set    = []
    heapq.heappush(open_set, (0, 0, start_state))
    came_from   = {}
    g_cost      = {start_state: 0}

    while open_set:
        _, current_g, current = heapq.heappop(open_set)
        cx, cy, ct = current

        # 목표 도달
        if (cx, cy) == goal:
            # 경로 역추적
            path = []
            node = current
            while node in came_from:
                path.append((node[0], node[1]))
                node = came_from[node]
            path.append(start)
            path.reverse()
            return path

        if ct >= max_time:
            continue

        if current_g > g_cost.get(current, float('inf')):
            continue

        for (dx, dy), cost in zip(directions, dir_costs):
            nx, ny, nt = cx+dx, cy+dy, ct+1

            if not is_valid(nx, ny):
                continue

            # 충돌 확인
            if reservation.is_vertex_conflict(nx, ny, nt):
                continue
            if reservation.is_edge_conflict(cx, cy, nx, ny, nt):
                continue

            neighbor = (nx, ny, nt)
            new_g    = g_cost[current] + cost

            if new_g < g_cost.get(neighbor, float('inf')):
                g_cost[neighbor]    = new_g
                came_from[neighbor] = current
                f = new_g + heuristic(nx, ny)
                heapq.heappush(open_set, (f, new_g, neighbor))

    return None  # 경로 없음


print("✅ ReservationTable + 충돌 회피 A* 정의 완료!")

✅ ReservationTable + 충돌 회피 A* 정의 완료!


In [5]:
# 다중 AMR 시뮬레이션
def run_multi_amr_simulation(grid_map, port_locations,
                              n_amrs=5, seed=42):
    """
    AMR N대 동시 경로 탐색 (충돌 회피 포함)
    """
    random.seed(seed)
    port_names = list(port_locations.keys())

    # 반송 요청 생성
    requests = []
    for i in range(n_amrs):
        s_name = random.choice(port_names)
        g_name = random.choice(port_names)
        while g_name == s_name:
            g_name = random.choice(port_names)

        s = port_locations[s_name]
        g = port_locations[g_name]
        requests.append({
            "amr_id":     i + 1,
            "start":      (s['grid_x'], s['grid_y']),
            "goal":       (g['grid_x'], g['grid_y']),
            "start_name": s_name,
            "goal_name":  g_name,
        })

    # Reservation Table 초기화
    reservation = ReservationTable()
    results     = []

    print(f"AMR {n_amrs}대 충돌 회피 경로 탐색 중...\n")

    for req in requests:
        t0   = time.time()
        path = astar_with_reservation(
            grid_map, req['start'], req['goal'],
            reservation, req['amr_id']
        )
        t1   = time.time()

        if path:
            reservation.reserve(path, req['amr_id'])
            status = f"✅ {len(path)}칸"
        else:
            status = "❌ 경로 없음"

        results.append({
            "amr_id":     req['amr_id'],
            "start":      req['start'],
            "goal":       req['goal'],
            "start_name": req['start_name'],
            "goal_name":  req['goal_name'],
            "path":       path,
            "path_length": len(path) if path else 0,
        })

        print(f"  AMR {req['amr_id']}: "
              f"{req['start_name'][:12]:12s} → "
              f"{req['goal_name'][:12]:12s} | "
              f"{status}  ({(t1-t0)*1000:.0f}ms)")

    # 충돌 검사
    n_conflicts = check_conflicts(results)
    print(f"\n충돌 검사: {n_conflicts}건")
    if n_conflicts == 0:
        print("✅ 충돌 없음!")
    else:
        print("⚠️ 충돌 발생 - 확인 필요")

    return results


def check_conflicts(results):
    """경로 간 충돌 검사"""
    conflicts = 0

    # 각 AMR의 시간별 위치
    time_pos = defaultdict(dict)  # {t: {amr_id: (x,y)}}
    for r in results:
        if r['path']:
            for t, pos in enumerate(r['path']):
                time_pos[t][r['amr_id']] = pos

    # Vertex conflict 검사
    for t, positions in time_pos.items():
        pos_list = list(positions.items())
        for i in range(len(pos_list)):
            for j in range(i+1, len(pos_list)):
                id1, pos1 = pos_list[i]
                id2, pos2 = pos_list[j]
                if pos1 == pos2:
                    conflicts += 1
                    print(f"  ⚠️ Vertex conflict: "
                          f"AMR{id1} & AMR{id2} "
                          f"at {pos1} t={t}")

    return conflicts


# 실행
results = run_multi_amr_simulation(
    grid_map, port_locations, n_amrs=5, seed=42
)

AMR 5대 충돌 회피 경로 탐색 중...

  AMR 1: ECH0179      → DIE0144      | ✅ 111칸  (31ms)
  AMR 2: ECH0514      → ECH0452      | ✅ 43칸  (1ms)
  AMR 3: ECH0408      → ECH0236      | ✅ 61칸  (54ms)
  AMR 4: ECH0160      → ECH0129      | ✅ 23칸  (8ms)
  AMR 5: LITH0282     → DIE0158      | ✅ 108칸  (17ms)

충돌 검사: 0건
✅ 충돌 없음!


In [6]:
# 애니메이션
def animate_simulation(grid_map, results, speed=100):
    """AMR 이동 애니메이션"""

    grid_h, grid_w = grid_map.shape

    # 최대 경로 길이
    max_t = max(
        len(r['path']) for r in results if r['path']
    )

    # AMR 색상
    amr_colors = [
        [1.0, 0.3, 0.3],  # 빨강
        [0.3, 0.6, 1.0],  # 파랑
        [0.2, 0.8, 0.3],  # 초록
        [1.0, 0.7, 0.1],  # 주황
        [0.8, 0.3, 0.9],  # 보라
    ]

    # 배경 맵
    color_map_vis = {
        0: [0.95, 0.95, 0.95],
        1: [0.20, 0.40, 0.70],
        2: [0.75, 0.90, 0.75],
    }
    bg = np.array([[color_map_vis[int(grid_map[r][c])]
                    for c in range(grid_w)]
                   for r in range(grid_h)], dtype=float)

    # 경로 범위 계산 (확대)
    all_xs, all_ys = [], []
    for r in results:
        if r['path']:
            all_xs += [p[0] for p in r['path']]
            all_ys += [p[1] for p in r['path']]

    margin = 15
    x1 = max(0, min(all_xs) - margin)
    x2 = min(grid_w, max(all_xs) + margin)
    y1 = max(0, min(all_ys) - margin)
    y2 = min(grid_h, max(all_ys) + margin)

    fig, ax = plt.subplots(figsize=(12, 7))
    im = ax.imshow(bg[y1:y2, x1:x2],
                   origin='lower', aspect='equal')

    # AMR 마커
    scatters = []
    for i, r in enumerate(results):
        if r['path']:
            c = amr_colors[i % len(amr_colors)]
            sc = ax.scatter([], [], c=[c], s=120,
                            zorder=5, edgecolors='white',
                            linewidths=1.5)
            scatters.append((i, sc, r))

    # AMR 번호 텍스트
    texts = []
    for i, sc, r in scatters:
        t = ax.text(0, 0, str(r['amr_id']),
                    ha='center', va='center',
                    fontsize=7, fontweight='bold',
                    color='white', zorder=6, visible=False)
        texts.append(t)

    title = ax.set_title("Step 0")
    step_n = [0]

    def update(frame):
        t = frame % max_t
        step_n[0] = t

        for idx, (i, sc, r) in enumerate(scatters):
            path = r['path']
            if t < len(path):
                px, py = path[t]
                sc.set_offsets([[px - x1, py - y1]])
                texts[idx].set_position((px-x1, py-y1))
                texts[idx].set_visible(True)
            else:
                # 목적지에 머물기
                px, py = path[-1]
                sc.set_offsets([[px - x1, py - y1]])
                texts[idx].set_position((px-x1, py-y1))

        title.set_text(
            f"다중 AMR 시뮬레이션  |  Step {t}/{max_t-1}"
        )
        return [im, title] + [sc for _,sc,_ in scatters] + texts

    ani = animation.FuncAnimation(
        fig, update,
        frames=max_t + 10,
        interval=speed,
        blit=True
    )
    plt.tight_layout()
    plt.close()
    return HTML(ani.to_jshtml())


# 애니메이션 실행
animate_simulation(grid_map, results, speed=150)

Output hidden; open in https://colab.research.google.com to view.

In [7]:
# 결과 저장
sim_result = [
    {
        "amr_id":      r['amr_id'],
        "start":       list(r['start']),
        "goal":        list(r['goal']),
        "path_length": r['path_length'],
        "path_found":  r['path'] is not None,
        "path": [list(p) for p in r['path']] if r['path'] else [],
    }
    for r in results
]

with open(f'{OUTPUT_DIR}/simulation_result.json', 'w') as f:
    json.dump(sim_result, f, indent=2)

# ── 체크리스트 ──
print("=" * 45)
print("4단계 체크리스트")
print("=" * 45)

n_success   = sum(1 for r in results if r['path'])
n_conflicts = check_conflicts(results)

checks = [
    ("ReservationTable 구현",
     True, ""),
    ("충돌 회피 A* 구현",
     True, ""),
    (f"AMR {len(results)}대 경로 탐색",
     n_success == len(results),
     f"{n_success}/{len(results)} 성공"),
    ("충돌 없음",
     n_conflicts == 0,
     f"{n_conflicts}건"),
    ("simulation_result.json 저장",
     os.path.exists(f'{OUTPUT_DIR}/simulation_result.json'),
     ""),
]

for name, result, detail in checks:
    status     = "✅" if result else "❌"
    detail_str = f"  ({detail})" if detail else ""
    print(f"  {status} {name}{detail_str}")

print()
if all(r for _, r, _ in checks):
    print("🎉 4단계 완료!")
    print("   → 5단계 반송 요청 큐로 이동!")
else:
    print("❌ 일부 실패. 확인 후 재실행하세요.")

4단계 체크리스트
  ✅ ReservationTable 구현
  ✅ 충돌 회피 A* 구현
  ✅ AMR 5대 경로 탐색  (5/5 성공)
  ✅ 충돌 없음  (0건)
  ✅ simulation_result.json 저장

🎉 4단계 완료!
   → 5단계 반송 요청 큐로 이동!
